# Where the data came from, and what was left out

Companion to `wiki_polis_analysis.ipynb`. Nothing here is needed to read the
findings — this is the audit trail: which queries pulled the data, what was
deliberately excluded, and the checks that ran before any of it was analysed.

## Two databases

wiki-polis splits its data by concern:

| | Database | Holds |
|---|---|---|
| **App** | MariaDB on Toolforge | conversations, who joined, pseudonyms, and **statement provenance** — which statement was submitted as an improvement on which other |
| **Polis** | PostgreSQL on the VPS | statements, every vote, and `math_main` — the clustering the Polis maths service computed |

Neither is read directly. Two exporters run *on the servers* and write a
de-identified bundle; only that bundle travels. The halves join on `person_key`, an
HMAC of the participant's internal id under a salt that never leaves the server.

The queries below are printed from the exporter modules, not copied, so what appears
here is necessarily what ran.

In [ ]:
import sys, textwrap
sys.path.insert(0, '.')
import export_polis_bundle as polis_export
import export_app_bundle as app_export

for name in ('Q_CONVERSATION', 'Q_COMMENTS', 'Q_VOTES_LATEST', 'Q_VOTES_HISTORY',
             'Q_PARTICIPANTS', 'Q_IDENTITY', 'Q_MATH_MAIN'):
    print(f'── {name} ' + '─' * (66 - len(name)))
    print(textwrap.dedent(getattr(polis_export, name)).strip())
    print()

Two notes.

**`Q_VOTES_LATEST` reads `votes_latest_unique`, not `votes`.** Polis keeps `votes` as
an append-only log — changing your mind inserts another row — and maintains
`votes_latest_unique` as current state. Counting the log would count people twice.

**`Q_IDENTITY` never reaches the bundle.** It maps each participant to the subject
Particiapi knows them by; the exporter uses it in memory to derive `person_key` and
discards it. Participants shown as `anon-…` have no such record and are deliberately
*not* joinable rather than guessed at.

`Q_COMMENTS` selects the author's `pid`, but the exporter drops it before writing. An
author column is harmless alone; combined with the pseudonym table it would
reconstruct who wrote what.

In [ ]:
print('app-side columns written to the bundle:\n')
for name in ('CONVERSATION_COLUMNS', 'FEATURED_COLUMNS', 'PROVENANCE_COLUMNS',
             'SIMILARITY_COLUMNS', 'PEOPLE_COLUMNS'):
    print(f'  {name:<22} {", ".join(getattr(app_export, name))}')

import bundle
print('\nnever exported in any mode:\n')
print(textwrap.fill(', '.join(sorted(bundle.DENIED_COLUMNS)), 78,
                    initial_indent='  ', subsequent_indent='  '))

## Checks that ran before analysis

These run identically on a synthetic test bundle, a local test conversation and real
server data. A check that passes locally and fails here indicates a problem in the
*data*, not in the analysis.

The last one rebuilds "latest vote per person per statement" from the append-only log
and compares it with the table Polis maintains for that purpose. Those can drift —
the Polis schema ships a repair query for exactly this — and every vote count depends
on them agreeing.

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import pipeline as P

b = P.load_bundles('2026-nlwiki-arbcom_bundle')
for check in P.integrity_checks(b):
    print(check)

## Personal information

Audited independently of the exporter's own self-check — that check passed on an
earlier bundle which reconstructed statement authorship through a join, so re-running
it would only re-confirm the same blind spot.

In [ ]:
!./.venv/bin/python audit_bundle.py 2026-nlwiki-arbcom_bundle effeietsanders ciell